# Module 24 — Data Drift Detection: PSI & Model Degradation Monitoring

> **Track 2 · Revolut Fintech Specialization**  
> **Difficulty**: Intermediate → Advanced  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Pandas, NumPy, Scikit-learn, Plotly  
> **Prerequisite**: Module 12 (Isolation Forest), Module 22 (Credit Risk Scoring)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Data Over Time](#3-synthetic-data-over-time)
4. [PSI Calculation](#4-psi-calculation)
5. [Drift Threshold Monitoring](#5-drift-threshold-monitoring)
6. [Model Performance Degradation](#6-model-performance-degradation)
7. [Visualization Dashboard](#7-visualization-dashboard)
8. [Retraining Recommendations](#8-retraining-recommendations)
9. [Key Business Takeaway](#9-key-business-takeaway)

---
## §1 · Business Context

### Why monitor data drift?

Models degrade as **data distributions shift** over time:
- **Concept drift**: Relationship between features and target changes.
- **Data drift**: Input feature distributions change.
- **Model staleness**: Performance degrades without retraining.

**Population Stability Index (PSI)**:
- Measures difference between two distributions.
- PSI < 0.1: No significant drift.
- 0.1 ≤ PSI < 0.2: Moderate drift, investigate.
- PSI ≥ 0.2: Significant drift, retrain model.

**Production monitoring**:
1. **Feature-level PSI**: Track each feature's distribution.
2. **Performance metrics**: Monitor AUC, precision, recall over time.
3. **Prediction distribution**: Watch for shifts in predicted probabilities.
4. **Business metrics**: Track actual fraud rates, approval rates.

In this notebook we will:
1. Generate synthetic data with gradual drift.
2. Calculate PSI for multiple features over time.
3. Monitor model performance degradation.
4. Build a drift monitoring dashboard.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Data Over Time

In [ ]:
# Generate synthetic data across multiple months
N_MONTHS = 12
N_SAMPLES_PER_MONTH = 10000

# Create data with drift
dfs = []
for month in range(N_MONTHS):
    # Simulate gradual drift
    drift_factor = month / N_MONTHS
    
    data = {
        'month': month + 1,
        'amount': np.random.exponential(100 + 20 * drift_factor, N_SAMPLES_PER_MONTH),
        'age': np.random.normal(35 + 2 * drift_factor, 10, N_SAMPLES_PER_MONTH),
        'credit_score': np.random.normal(650 - 10 * drift_factor, 100, N_SAMPLES_PER_MONTH),
        'income': np.random.lognormal(10.5, 0.8, N_SAMPLES_PER_MONTH),
        'txn_count': np.random.poisson(10 + 3 * drift_factor, N_SAMPLES_PER_MONTH),
    }
    dfs.append(pd.DataFrame(data))

df_all = pd.concat(dfs, ignore_index=True)

print(f"✓ Generated data over {N_MONTHS} months:")
print(f"  Total samples: {len(df_all):,}")
print(f"  Samples per month: {N_SAMPLES_PER_MONTH:,}")
print(f"\nDrift scenario:")
print(f"  - Transaction amounts gradually increase")
print(f"  - Customer age distribution shifts older")
print(f"  - Credit scores gradually decrease")

---
## §4 · PSI Calculation

In [ ]:
# Calculate Population Stability Index (PSI)
def calculate_psi(reference, current, bins=10):
    """Calculate PSI between two distributions."""
    # Bin the reference distribution
    breakpoints = np.percentile(reference, np.linspace(0, 100, bins + 1))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf
    
    # Count proportions
    ref_counts = np.histogram(reference, bins=breakpoints)[0] / len(reference)
    curr_counts = np.histogram(current, bins=breakpoints)[0] / len(current)
    
    # Avoid division by zero
    ref_counts = np.where(ref_counts == 0, 0.0001, ref_counts)
    curr_counts = np.where(curr_counts == 0, 0.0001, curr_counts)
    
    # Calculate PSI
    psi = np.sum((curr_counts - ref_counts) * np.log(curr_counts / ref_counts))
    return psi

# Use month 1 as reference
reference_data = df_all[df_all['month'] == 1]

# Calculate PSI for each feature over time
features = ['amount', 'age', 'credit_score', 'income', 'txn_count']
psi_over_time = {feat: [] for feat in features}

for month in range(2, N_MONTHS + 1):
    current_data = df_all[df_all['month'] == month]
    for feat in features:
        psi = calculate_psi(reference_data[feat].values, current_data[feat].values)
        psi_over_time[feat].append(psi)

print("✓ PSI calculated for all features over time")
print("\nPSI Interpretation:")
print("  - PSI < 0.1: No significant drift")
print("  - 0.1 ≤ PSI < 0.2: Moderate drift")
print("  - PSI ≥ 0.2: Significant drift, retraining recommended")

---
## §5 · Drift Threshold Monitoring

In [ ]:
# Define drift thresholds
PSI_WARNING = 0.1
PSI_CRITICAL = 0.2

# Check drift status for latest month
print("=" * 70)
print("DRIFT MONITORING STATUS (Month 12)")
print("=" * 70)

for feat in features:
    latest_psi = psi_over_time[feat][-1]
    if latest_psi >= PSI_CRITICAL:
        status = "🔴 CRITICAL"
    elif latest_psi >= PSI_WARNING:
        status = "🟡 WARNING"
    else:
        status = "🟢 OK"
    
    print(f"\n{feat}:")
    print(f"  PSI: {latest_psi:.4f}")
    print(f"  Status: {status}")

---
## §6 · Model Performance Degradation

In [ ]:
# Simulate model performance over time
# Train baseline model on month 1
reference_data = df_all[df_all['month'] == 1].copy()
reference_data['target'] = np.random.binomial(1, 0.05, len(reference_data))

feature_cols = ['amount', 'age', 'credit_score', 'income', 'txn_count']
X_ref = reference_data[feature_cols]
y_ref = reference_data['target']

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_ref, y_ref)

# Evaluate on each month
performance_over_time = []
for month in range(1, N_MONTHS + 1):
    month_data = df_all[df_all['month'] == month].copy()
    month_data['target'] = np.random.binomial(1, 0.05 + 0.02 * (month - 1) / N_MONTHS, len(month_data))
    
    X_month = month_data[feature_cols]
    y_month = month_data['target']
    
    y_prob = model.predict_proba(X_month)[:, 1]
    auc = roc_auc_score(y_month, y_prob)
    performance_over_time.append(auc)

print("✓ Model performance tracked over time")
print(f"\nPerformance degradation:")
print(f"  Month 1 AUC: {performance_over_time[0]:.4f}")
print(f"  Month 12 AUC: {performance_over_time[-1]:.4f}")
print(f"  Degradation: {(performance_over_time[0] - performance_over_time[-1])*100:.2f}%")

---
## §7 · Visualization Dashboard

In [ ]:
# Create drift monitoring dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['PSI Over Time', 'Model Performance', 'Drift Status', 'Feature Distributions']
)

# PSI over time
for feat in features:
    fig.add_trace(go.Scatter(
        x=list(range(2, N_MONTHS + 1)),
        y=psi_over_time[feat],
        name=feat,
        mode='lines+markers'
    ), row=1, col=1)

fig.add_hline(y=PSI_WARNING, line_dash="dash", line_color="yellow", row=1, col=1)
fig.add_hline(y=PSI_CRITICAL, line_dash="dash", line_color="red", row=1, col=1)

# Model performance
fig.add_trace(go.Scatter(
    x=list(range(1, N_MONTHS + 1)),
    y=performance_over_time,
    name='AUC',
    mode='lines+markers',
    line=dict(color='blue', width=3)
), row=1, col=2)

# Drift status heatmap
status_data = []
for feat in features:
    latest_psi = psi_over_time[feat][-1]
    status_data.append(1 if latest_psi >= PSI_CRITICAL else (0.5 if latest_psi >= PSI_WARNING else 0))

fig.add_trace(go.Bar(
    x=features,
    y=status_data,
    marker_color=['red' if s >= 1 else ('yellow' if s >= 0.5 else 'green') for s in status_data],
    name='Drift Status'
), row=2, col=1)

# Feature distribution comparison
ref_amounts = reference_data['amount']
curr_amounts = df_all[df_all['month'] == N_MONTHS]['amount']

fig.add_trace(go.Histogram(x=ref_amounts, name='Month 1', opacity=0.7), row=2, col=2)
fig.add_trace(go.Histogram(x=curr_amounts, name='Month 12', opacity=0.7), row=2, col=2)

fig.update_layout(height=600, showlegend=True)
fig.show()

print("💡 Dashboard shows:")
print("   - PSI trends for each feature")
print("   - Model performance degradation")
print("   - Current drift status")
print("   - Distribution shifts over time")

---
## §8 · Retraining Recommendations

In [ ]:
# Generate retraining recommendations
print("=" * 70)
print("RETRAINING RECOMMENDATIONS")
print("=" * 70)

recommendations = []
for feat in features:
    latest_psi = psi_over_time[feat][-1]
    if latest_psi >= PSI_CRITICAL:
        recommendations.append(f"🔴 {feat}: PSI={latest_psi:.4f} - RETRAIN IMMEDIATELY")
    elif latest_psi >= PSI_WARNING:
        recommendations.append(f"🟡 {feat}: PSI={latest_psi:.4f} - Monitor closely")
    else:
        recommendations.append(f"🟢 {feat}: PSI={latest_psi:.4f} - OK")

print("\n".join(recommendations))

print("\n💡 Production monitoring checklist:")
print("   1. Calculate PSI daily/weekly for key features")
print("   2. Set up alerts for PSI > 0.1 (warning) and > 0.2 (critical)")
print("   3. Track model performance metrics over time")
print("   4. Retrain model when multiple features show drift")
print("   5. A/B test new model against production model")

---
## §9 · Key Business Takeaway

> ### 🎯 Action Items for ML Ops Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Production model monitoring, retraining decisions |
> | **Metrics** | PSI for features, AUC for performance |
> | **Thresholds** | PSI > 0.1 warning, > 0.2 critical |
> | **Frequency** | Daily PSI, weekly performance review |
> | **Action** | Retrain when PSI > 0.2 or AUC drops > 5% |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Monitor all production models**:
>    ```
>    Daily: Calculate PSI for input features
>    Weekly: Review model performance metrics
>    Monthly: Full model audit and potential retraining
>    ```
>
> 2. **Drift handling strategies**:
>    | Drift Type | Detection | Response |
>    |------------|-----------|----------|
>    | Gradual | PSI trending up | Schedule retraining |
>    | Sudden | PSI spike | Investigate + emergency retrain |
>    | Seasonal | Regular patterns | Adjust for seasonality |
>
> 3. **Automated retraining pipeline**:
>    - Trigger retraining when PSI > 0.2 for 3+ features
>    - A/B test new model vs. production
>    - Gradual rollout (10% → 50% → 100%)
>
> ### 🔑 Key Takeaways
>
> 1. **PSI is the standard metric** for detecting data drift.
> 2. **Monitor both features and performance** for comprehensive tracking.
> 3. **Set up automated alerts** for drift thresholds.
> 4. **Regular retraining** is essential for production models.
> 5. **A/B test new models** before full deployment.